In [3]:
import pandas as pd
from collections import defaultdict
import numpy as np
import json

In [4]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

articles = load_jsonl("data/articles.jsonl")
summaries = load_jsonl("data/summaries.jsonl")

In [5]:
articles_dict = {a["article_id"]: a for a in articles}

grouped_summaries = defaultdict(list)

for s in summaries:
    grouped_summaries[s["article_id"]].append(s)

In [6]:
# regarder 1 exemple complet
aid = articles[0]["article_id"]

print("TITLE:", articles_dict[aid]["title"])
print("\nARTICLE:\n", articles_dict[aid]["text"][:1000])

print("\nREFERENCE:\n", articles_dict[aid]["reference_summary"])

print("\nSUMMARIES:\n")

for i, s in enumerate(grouped_summaries[aid]):
    print(f"\n--- Summary {i} ---")
    print(s["summary"])

TITLE: イスラエル入植地「違法ではない」 米政府が方針転換

ARTICLE:
 イスラエルは東エルサレムを含むヨルダン川西岸に、100カ所以上のユダヤ人入植地を建設してきた マイク・ポンペオ国務長官が記者会見で、ヨルダン川西岸の地位は、イスラエルとパレスチナが交渉すべきものとの見解を示した。 「アメリカは法的論争をすべての面から慎重に検討した結果、イスラエル民間人によるヨルダン川西岸の入植地建設はただちに、国際法に則っていないというわけではないとの結論に達した」 ポンペオ氏はまた、「民間人の入植地建設を国際法に則していないと指摘する方法では、うまくいかなかった。平和の促進につながらなかった」と述べた。 「歴史の過ちを正す」 イスラエルは、アメリカによるこの方針転換を歓迎した。 ベンヤミン・ネタニヤフ首相は、ドナルド・トランプ米政権による方針転換は「歴史の過ちを正す」ものだと評価。他国も続くよう求めた。 イスラエルのネタニヤフ首相は、入植地を手放すことは決してないとしている 一方、パレスチナ解放機構（PLO）の交渉担当者サエブ・エレカット氏は、アメリカの決定を「世界の安定と安全、平和」を危険にさらすものと批判。 国際法を「ジャングル法」に変えてしまう恐れがあると述べた。 140の入植地に60万人 イスラエルによるパレスチナ自治区での入植地建設は長年、同国と国際社会、パレスチナの間における論争の原因となっている。 イスラエルは1967年の第3次中東戦争後、入植地を建設してきた。現在、約140の入植地があり、計約60万人のユダヤ人が居住している。 入植地は一般的に、国際法に照らして違法だと考えられている。ただ、イスラエルは一貫してこの見方に異を唱えている。 パレスチナは長年、すべての入植地の撤廃を要求してきた。将来、独立したパレスチナ国家となるべき土地に入植地があると、国家建設がほぼ実現不可能になると主張している。 オバマ政権はイスラエル擁護せず アメリカは1978年、ジミー・カーター政権で、イスラエルの入植地建設は国際法違反と結論づけた。しかし1981年には、ロナルド・レーガン大統領がこの結論に不同意を表明。入植地が固有の違法性をもつとは思わないと述べた。 以来、アメリカは入植地を「違法」ではなく「非合法」と表現してきた。さらに、国連でイスラエルが非難されるのを

In [ ]:
def rule_features(summary, article_text):
    sentences = summary.split("。")  # japonais sentence split approx

    num_sentences = len([s for s in sentences if s.strip()])
    length_ratio = len(summary) / max(1, len(article_text))

    return {
        "num_sentences": num_sentences,
        "length_ratio": length_ratio
    }

In [ ]:
def rule_penalty(features):
    penalty = 0

    if features["num_sentences"] > 3:
        penalty += 1

    if features["length_ratio"] < 0.01:
        penalty += 0.5

    if features["length_ratio"] > 0.3:
        penalty += 1

    return penalty

In [11]:
import openai

def llm_judge(article, summary):
    prompt = f"""
You are an expert evaluator of Japanese news summaries.

Evaluate the summary based on the article.

Return JSON with:
- faithfulness (1-5)
- coverage (1-5)
- conciseness (1-5)
- fluency (1-5)

ARTICLE:
{article}

SUMMARY:
{summary}

Return ONLY JSON.
"""

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return json.loads(response["choices"][0]["message"]["content"])

In [ ]:
def compute_score(llm_scores, penalty):
    score = (
        0.4 * llm_scores["faithfulness"] +
        0.3 * llm_scores["coverage"] +
        0.2 * llm_scores["conciseness"] +
        0.1 * llm_scores["fluency"]
    )

    return score - penalty

In [ ]:
results = []

for s in summaries:
    article = articles_dict[s["article_id"]]

    features = rule_features(s["summary"], article["text"])
    penalty = rule_penalty(features)

    llm_scores = llm_judge(article["text"], s["summary"])

    final_score = compute_score(llm_scores, penalty)

    results.append({
        "summary_id": s["summary_id"],
        "article_id": s["article_id"],
        "score": final_score,
        **llm_scores,
        **features
    })

APIRemovedInV1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742
